In [1]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from moviepy.video.io.VideoFileClip import VideoFileClip
import os
import glob

In [ ]:
model_path = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
    _attn_implementation="eager"
).to("cpu")

model.eval()


SmolVLMForConditionalGeneration(
  (model): SmolVLMModel(
    (vision_model): SmolVLMVisionTransformer(
      (embeddings): SmolVLMVisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): SmolVLMEncoder(
        (layers): ModuleList(
          (0-11): 12 x SmolVLMEncoderLayer(
            (self_attn): SmolVLMVisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): SmolVLMVisionMLP(
              (activation_fn): PytorchGELUTanh()
              (fc1): Linear(in_features=768, out_

In [ ]:
SYSTEM = """
You are a precise video event detection model.

Rules:
- Identify distinct events
- Do not repeat events
- Keep descriptions short
- No explanations
"""


PROMPT_EVENT_ONLY = """
List all salient events in the video.

Return detected events in the following way: Salient Event: <event description> 



Rules:
- Each event must be short
- Events must be chronologically ordered
- No repetition
- No extra text
"""


# SYSTEM = (
#     "You are an event detection model. "
#     "Your task is to identify only the salient visible events in the video. "
#     "Do not describe background details unless they are necessary for understanding an event. "
#     "Do not repeat the same event. "
#     "Return the answer as a numbered list."
# )

#1) Retrieve all salient events in the video. Shit for both video 5 and video 6!
#2) Describe all the key events in the video. Much better for video 5 and incredibly better for video 6. A lot of yapping though and overlap.
#3) "Describe all the key events in the video." "Return only events that are important for understanding the video." A bit better, but still hasn't captured salient events.

# PROMPT_EVENT_ONLY = ("""
# You are analyzing a low-quality video made of sampled frames.

# Your task is to identify key events events: important visible actions or changes over time.
# Focus on what changes between frames, not on static background details.   
# """)

PROMPT_EVENT_TIMESTAMP = (
"""
Describe all the key events in the video with approximate timestamps.
"""
)

# check function
def run_vlm_on_video(video_path, prompt, max_frames=8, max_new_tokens=500):
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        video_load_backend="pyav",
        max_frames=max_frames,
    )

    inputs = {
        k: v.to("cpu") if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens
        )

    generated_only = generated_ids[:, inputs["input_ids"].shape[1]:]

    answer = processor.batch_decode(
        generated_only,
        skip_special_tokens=True
    )[0]

    return answer.strip()



In [4]:
def split_video_into_chunks(video_path, output_dir, chunk_length=8):
    os.makedirs(output_dir, exist_ok=True)

    # If chunks already exist, reuse them
    existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))
    if existing_chunks:
        chunk_paths = []

        for chunk_path in existing_chunks:
            filename = os.path.basename(chunk_path)
            parts = filename.replace("chunk_", "").replace(".mp4", "").split("_")
            start = int(parts[0])
            end = int(parts[1])
            chunk_paths.append((chunk_path, start, end))

        print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")
        return chunk_paths

    # Otherwise, create chunks
    video = VideoFileClip(video_path)
    duration = int(video.duration)

    chunk_paths = []

    for start in range(0, duration, chunk_length):
        end = min(start + chunk_length, duration)

        chunk_path = os.path.join(
            output_dir,
            f"chunk_{start:04d}_{end:04d}.mp4"
        )

        if hasattr(video, "subclipped"):
            clip = video.subclipped(start, end)
        else:
            clip = video.subclip(start, end)

        clip.write_videofile(
            chunk_path,
            codec="libx264",
            audio=False,
            logger=None
        )

        clip.close()
        chunk_paths.append((chunk_path, start, end))

    video.close()
    return chunk_paths

In [5]:
video_path = "TVSum/video_6.mp4"

chunk_length = 8
output_dir = f"video_6_chunks_{chunk_length}s"

chunks = split_video_into_chunks(
    video_path,
    output_dir=output_dir,
    chunk_length= chunk_length# changed to 8 seconds
)

all_outputs = []

for chunk_path, start, end in chunks:
    output = run_vlm_on_video(
        chunk_path,
        PROMPT_EVENT_ONLY,
        max_frames=16, # 2 frames per seconds
        max_new_tokens=500 # match it to length of 1-2 sentences
    )

    all_outputs.append({
        "chunk": chunk_path,
        "start": start,
        "end": end,
        "vlm_output": output
    })

    print(f"\nCHUNK {start}-{end} seconds")
    print(output)

Reusing 41 existing chunks from video_6_chunks_8s

CHUNK 0-8 seconds
The video shows a person standing in front of a wooden table, holding a tennis ball. The person is wearing a green shirt and is holding the tennis ball in their right hand. The table is surrounded by chairs and a fence. The person is standing in front of the table, holding the tennis ball.

CHUNK 8-16 seconds
The video shows a person standing in front of a wooden table, holding a green bucket. The person is wearing a blue shirt and is standing in front of a wooden fence. The table is surrounded by chairs and a green bucket.

CHUNK 16-24 seconds
The video shows a car driving down a road, passing by a house and trees. The car is driving on the left side of the road, and the house is on the right side. The car is driving on the left side of the road, and the house is on the right side. The car is driving on the left side of the road, and the house is on the right side. The car is driving on the left side of the road, and

In [6]:
import re

def parse_vlm_events(text):
    """
    Extracts events from VLM output formatted like:
    Salient event 1: A person enters the room
    Salient event 2: A person sits down, 00:03-00:07
    """

    pattern = r"Salient event\s*\d+\s*:\s*(.*?)(?=\n\s*Salient event\s*\d+\s*:|\Z)"
    matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)

    parsed_events = []

    for match in matches:
        event_text = match.strip().replace("\n", " ")

        time_pattern = r"(\d{1,2}:\d{2})\s*[-–]\s*(\d{1,2}:\d{2})"
        time_match = re.search(time_pattern, event_text)

        if time_match:
            start_time = time_match.group(1)
            end_time = time_match.group(2)
            event_description = re.sub(time_pattern, "", event_text).strip(" ,.-")
        else:
            start_time = None
            end_time = None
            event_description = event_text.strip(" ,.-")

        parsed_events.append({
            "event": event_description,
            "start_time": start_time,
            "end_time": end_time
        })

    return parsed_events

In [7]:
# parsed_event_only = parse_vlm_events(event_only_output)
# parsed_event_timestamp = parse_vlm_events(event_timestamp_output)

# print(parsed_event_only)
# print(parsed_event_timestamp)

In [8]:
predicted_events = []

for item in all_outputs:
    parsed = parse_vlm_events(item["vlm_output"])
    predicted_events.extend([e["event"] for e in parsed])



# need deduplication at some point (probably earlier)